# CRL focus explorer

**Interface revision: separate parameter and results cells · plane-wave input · simultaneous 2D profiles · synchronous z navigation · calculation progress and stage ETA.**

1. Run **1 · Prepare XPOSim**.
2. Run **2 · CRL and source parameters**. The editable lens and source fields appear directly below that cell and remain there.
3. Run **3 · Results**, then press **Compute**. No optical calculation starts before that click. To change the lens or source later, edit the fields in cell 2 and press **Compute** again in cell 3.

**Run all** initializes both panels without calculating. For GPU: **Runtime → Change runtime type → GPU**, reconnect, and run all. The actual CUDA device is checked.


In [ ]:
#@title 1 · Prepare XPOSim
import hashlib
import importlib.util
import re
import subprocess
import sys
import urllib.request
from pathlib import Path

SOURCE_REF = '077082624064135ed28e3d9560ca5c61fc0206c0'
SOURCE_SHA256 = 'a73904acb6430a362c4f8d7f5add49b56871e2a573939d07a2b14e1744f3309c'
PROJECT_DIR = Path.cwd() / '.xposim_runtime' / SOURCE_REF
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
requirements = {'numpy':'numpy', 'scipy':'scipy', 'matplotlib':'matplotlib', 'ipywidgets':'ipywidgets>=8,<9'}
missing = [requirement for name,requirement in requirements.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir',*missing])
try:
    gpu_info = subprocess.run(['nvidia-smi'],capture_output=True,text=True,timeout=15)
except (FileNotFoundError,subprocess.TimeoutExpired):
    gpu_info = None
if gpu_info is not None and gpu_info.returncode==0 and importlib.util.find_spec('cupy') is None:
    try:
        toolkit = subprocess.run(['nvcc','--version'],capture_output=True,text=True,timeout=15)
        release = re.search(r'release\s+(\d+)\.',toolkit.stdout)
    except (FileNotFoundError,subprocess.TimeoutExpired):
        release = None
    if release is None:
        release = re.search(r'CUDA Version:\s*(\d+)\.',gpu_info.stdout)
    major = int(release.group(1)) if release else None
    wheel = {12:'cupy-cuda12x',13:'cupy-cuda13x'}.get(major)
    if wheel:
        try:
            subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir',f'{wheel}==14.2.0'])
            importlib.invalidate_caches()
        except subprocess.CalledProcessError:
            print('CuPy installation failed. Auto backend will use NumPy when CUDA is unavailable.')
    else:
        print('CUDA version was not recognized. CPU execution remains available.')
for relative in ['XPOSim.py','xray_material_data/AtomicConstants.dat',
                 'xray_material_data/CrossSec-Compton_McMaster.dat','xray_material_data/f1f2_Windt.dat']:
    target = PROJECT_DIR / relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists():
        url = f'https://raw.githubusercontent.com/mamichberdey/XPOSim/{SOURCE_REF}/{relative}'
        with urllib.request.urlopen(url,timeout=90) as response:
            content = response.read()
        temporary = target.with_suffix(target.suffix+'.download')
        temporary.write_bytes(content)
        temporary.replace(target)
if hashlib.sha256((PROJECT_DIR/'XPOSim.py').read_bytes()).hexdigest()!=SOURCE_SHA256:
    raise RuntimeError('XPOSim checksum mismatch. Remove .xposim_runtime and repeat preparation.')
spec = importlib.util.spec_from_file_location('XPOSim',PROJECT_DIR/'XPOSim.py')
xpos = importlib.util.module_from_spec(spec)
sys.modules['XPOSim'] = xpos
spec.loader.exec_module(xpos)
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
print(f'XPOSim {xpos.__version__}; available backend: {xpos.select_backend("auto")}.')
print('Ready. Run cell 2 to open CRL and source inputs, then cell 3 for Results. Press Compute only after editing the inputs.')


In [ ]:
#@title 2 · CRL and source parameters — run this cell to show the input fields
import html
import ipywidgets as widgets
from traitlets import TraitError
from IPython.display import display, HTML, clear_output
try:
    from google.colab import output as colab_output
except ImportError:
    colab_output = None
if colab_output is not None:
    colab_output.enable_custom_widget_manager()

_previous_values={name:widget.value for name,widget in globals().get('controls',{}).items()}
if _previous_values.get('accuracy') in {'Quick','Refined'}:
    _previous_values['accuracy']={'Quick':'Fast','Refined':'Accurate'}[_previous_values['accuracy']]
_first_explorer_initialization=not bool(_previous_values)
_previous_editor=globals().get('parameter_editor')
if _previous_editor is not None:
    _previous_editor.close()
for widget in globals().get('controls',{}).values():
    widget.unobserve_all()
    widget.close()
controls = {}
containers = {}
def input_field(name, label, widget, tooltip=''):
    if name in _previous_values:
        try:
            value=_previous_values[name]
            if name=='accuracy':value={'Quick':'Fast','Refined':'Accurate'}.get(value,value)
            if name=='z_mode':value={'Auto around numerical focus':'Auto around intensity peak'}.get(value,value)
            widget.value=value
        except (ValueError,TypeError,TraitError):
            pass
    controls[name] = widget
    widget.layout.width = '100%'
    label_html = widgets.HTML('<span style="font-size:12px;font-weight:500" title="' + html.escape(tooltip, quote=True) + '">' + html.escape(label) + '</span>')
    box = widgets.VBox([label_html, widget], layout=widgets.Layout(width='100%'))
    box.add_class('xposim-field-'+name)
    containers[name] = box
    return box

def group(fields):
    return widgets.GridBox(fields, layout=widgets.Layout(
        grid_template_columns='repeat(auto-fit, minmax(215px, 245px))', grid_gap='8px 14px', width='100%'))

optics = group([
    input_field('focusing', 'CRL focusing', widgets.Dropdown(options=['One-dimensional focusing', 'Two-dimensional focusing'])),
    input_field('grid', 'Calculation grid', widgets.Dropdown(options=['1D', '2D'])),
    input_field('accuracy', 'Sampling', widgets.Dropdown(options=['Fast', 'Accurate'],value='Fast')),
    input_field('material', 'Material', widgets.Dropdown(options=['Beryllium (Be)', 'Silicon (Si)', 'Aluminium (Al)', 'Diamond (C)', 'Custom'],value='Silicon (Si)')),
    input_field('formula', 'Chemical formula', widgets.Text(value='SiO2')),
    input_field('density_mode', 'Material density', widgets.Dropdown(options=['Auto', 'Manual'])),
    input_field('density', 'Density (g/cm³)', widgets.FloatText(value=2.33)),
    input_field('energy', 'Photon energy (keV)', widgets.FloatText(value=12.0)),
    input_field('radius', 'Surface radius (µm)', widgets.FloatText(value=6.25)),
    input_field('aperture', 'Full aperture (µm)', widgets.FloatText(value=50.0)),
    input_field('bridge', 'Bridge thickness (µm)', widgets.FloatText(value=2.0)),
    input_field('elements', 'CRL elements', widgets.BoundedIntText(value=26, min=1, max=10000)),
    input_field('depth', 'Structure depth along y (µm)', widgets.FloatText(value=50.0)),
])
source_controls = group([
    input_field('source', 'Source', widgets.Dropdown(options=['Point source', 'Gaussian source (incoherent)', 'Plane wave at CRL entrance'])),
    input_field('distance', 'Source-to-CRL distance (m)', widgets.FloatText(value=15.0)),
    input_field('fwhm_x', 'Source FWHM x (µm)', widgets.FloatText(value=5.0)),
    input_field('fwhm_y', 'Source FWHM y (µm)', widgets.FloatText(value=5.0)),
    input_field('source_method', 'Extended-source method', widgets.Dropdown(options=['Gaussian convolution', 'Incoherent point sampling'])),
    input_field('source_points', 'Source nodes per axis', widgets.BoundedIntText(value=7, min=3, max=101, step=2), 'Gaussian quadrature order. Increase it to check convergence.'),
])
advanced_fields = group([
    input_field('adaptive_window', 'Adapt propagation window', widgets.Checkbox(value=True, indent=False)),
    input_field('nx', 'Grid points x (0 = auto)', widgets.IntText(value=0)),
    input_field('ny', 'Grid points y (0 = auto)', widgets.IntText(value=0)),
    input_field('window', 'Initial window / geometry factor', widgets.FloatText(value=1.3)),
    input_field('batch', 'Batch size (0 = auto)', widgets.IntText(value=0)),
    input_field('z_mode', 'Displayed z ROI', widgets.Dropdown(options=['Auto around intensity peak', 'Manual bounds'])),
    input_field('z_min', 'z minimum from CRL exit (mm)', widgets.FloatText(value=25.0)),
    input_field('z_max', 'z maximum from CRL exit (mm)', widgets.FloatText(value=45.0)),
    input_field('z_points', 'Displayed z samples', widgets.BoundedIntText(value=51, min=5, max=501)),
    input_field('z_level', 'Axial ROI intensity fraction', widgets.BoundedFloatText(value=0.2, min=0.001, max=0.99, step=0.05)),
    input_field('xy_level', 'Transverse ROI intensity fraction', widgets.BoundedFloatText(value=0.01, min=0.00001, max=0.5, step=0.01)),
    input_field('tolerance', 'Peak search tolerance (µm; 0 = auto)', widgets.FloatText(value=0.0)),
])
advanced = widgets.Accordion(children=[advanced_fields], selected_index=None)
advanced.set_title(0, 'Sampling, ROI and search')

def sync_controls(change=None):
    two_focusing = controls['focusing'].value == 'Two-dimensional focusing'
    if two_focusing:
        controls['grid'].value = '2D'
    controls['grid'].disabled = two_focusing
    two_grid = controls['grid'].value == '2D'
    gaussian = controls['source'].value == 'Gaussian source (incoherent)'
    visibility = {
        'depth': two_grid and not two_focusing,
        'formula': controls['material'].value == 'Custom',
        'density': True,
        'distance': controls['source'].value != 'Plane wave at CRL entrance',
        'fwhm_x': gaussian, 'fwhm_y': gaussian and two_grid,
        'source_method': gaussian,
        'source_points': gaussian and controls['source_method'].value == 'Incoherent point sampling',
        'ny': two_grid,
        'z_min': controls['z_mode'].value == 'Manual bounds',
        'z_max': controls['z_mode'].value == 'Manual bounds',
    }
    presets={'Beryllium (Be)':1.848,'Silicon (Si)':2.33,'Aluminium (Al)':2.70,'Diamond (C)':3.515}
    custom=controls['material'].value=='Custom'
    if custom and controls['density_mode'].value!='Manual':
        controls['density_mode'].value='Manual'
    controls['density_mode'].disabled=custom
    automatic=controls['density_mode'].value=='Auto' and not custom
    controls['density'].disabled=automatic
    if automatic:
        controls['density'].value=presets[controls['material'].value]
    density_label='Density (g/cm³)'
    if automatic:
        density_label+=' · Auto: '+controls['material'].value
    containers['density'].children[0].value='<span style="font-size:12px;font-weight:500">'+html.escape(density_label)+'</span>'
    for name, visible in visibility.items():
        containers[name].layout.display = '' if visible else 'none'
for name in ['focusing', 'grid', 'source', 'source_method', 'material', 'density_mode', 'z_mode']:
    controls[name].observe(sync_controls, names='value')
sync_controls()

def read_configuration():
    v = {name: widget.value for name, widget in controls.items()}
    for name in (('nx', 'batch', 'tolerance', 'ny') if v['grid']=='2D' else ('nx', 'batch', 'tolerance')):
        if v[name] < 0:
            raise ValueError(f'{name} must be non-negative; zero selects automatic control.')
    presets = {'Beryllium (Be)': ('Be', 1.848), 'Silicon (Si)': ('Si', 2.33),
               'Aluminium (Al)': ('Al', 2.70), 'Diamond (C)': ('C', 3.515)}
    formula, automatic_density = presets.get(v['material'], (v['formula'], v['density']))
    density=automatic_density if v['density_mode']=='Auto' and v['material']!='Custom' else v['density']
    return xpos.CRLFocusConfig(
        energy_kev=v['energy'], material_formula=formula, density_g_cm3=density,
        radius_m=v['radius']*1e-6, aperture_m=v['aperture']*1e-6,
        bridge_thickness_m=v['bridge']*1e-6, n_elements=v['elements'],
        geometry='two-dimensional' if v['focusing']=='Two-dimensional focusing' else 'one-dimensional',
        grid_dimension=2 if v['grid']=='2D' else 1,
        structure_depth_m=v['depth']*1e-6,
        source_distance_m=None if v['source']=='Plane wave at CRL entrance' else v['distance'],
        source_type={'Point source':'point','Gaussian source (incoherent)':'gaussian','Plane wave at CRL entrance':'plane'}[v['source']],
        source_fwhm_x_m=v['fwhm_x']*1e-6 if v['source']=='Gaussian source (incoherent)' else 0.0,
        source_fwhm_y_m=v['fwhm_y']*1e-6 if v['source']=='Gaussian source (incoherent)' and v['grid']=='2D' else 0.0,
        source_method='points' if v['source_method']=='Incoherent point sampling' else 'convolution',
        source_points=v['source_points'],
        backend='auto',
        accuracy='preview' if v['accuracy']=='Fast' else 'refined',
        grid_points=v['nx'] or None, grid_points_y=(v['ny'] or None) if v['grid']=='2D' else None,
        window_factor=v['window'], adaptive_window=v['adaptive_window'], batch_size=v['batch'] or None,
        scan_points=v['z_points'],
        scan_range_m=None if v['z_mode']=='Auto around intensity peak' else (v['z_min']*1e-3,v['z_max']*1e-3),
        axial_roi_level=v['z_level'], transverse_roi_level=v['xy_level'],
        peak_search_tolerance_m=v['tolerance']*1e-6 if v['tolerance'] else None,
    )

form_body=widgets.VBox([widgets.HTML('<b>CRL</b>'),optics,
    widgets.HTML('<b>Source</b>'),source_controls,advanced],layout=widgets.Layout(width='100%'))
parameter_editor=widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 4px">CRL and source parameters</h3>'
                 '<div style="font-size:12px;margin-bottom:6px">Edit the inputs here, then press <b>Compute</b> in the Results cell below.</div>'),
    form_body],layout=widgets.Layout(width='100%',max_width='1280px',margin='0 auto'))
parameter_editor.add_class('xposim-parameters')
display(parameter_editor)
print('Parameter form ready. Next: run the Results cell, then press Compute. No calculation starts automatically.')


In [ ]:
#@title 3 · Results — press Compute after entering parameters above
import gc
import io
import time
from matplotlib.ticker import FuncFormatter, MaxNLocator
from matplotlib.colors import Normalize

if 'controls' not in globals() or 'parameter_editor' not in globals():
    raise RuntimeError('Run the CRL and source parameters cell above before Results.')

# Reinitialization preserves inputs and waits for an explicit calculation.
_previous_state=globals().get('state',{})
if _previous_state.get('plane_update_handle') is not None:
    _previous_state['plane_update_handle'].cancel()
for name in ('slider','z_input','previous_button','next_button','plane_label'):
    widget=_previous_state.get(name)
    if widget is not None:
        widget.unobserve_all()
        widget.close()
for name in ('explorer_root','compute_button','plane_image','scan_image','plane_view','progress_view'):
    old=globals().get(name)
    if old is not None:
        old.close()
state={'result':None,'slider':None,'handler':None,'plane_updates':0,'error':None,
       'z_unit_scale':1e-3,'scan_axes':None}
CMAP=plt.get_cmap('CMRmap')
INTENSITY_COLOR='#d43d3d'
PROFILE_COLOR='#3977d7'
PROFILE_COLORS=(PROFILE_COLOR,'#8b5ac7')
WIDTH_COLORS=('#25864b','#62ad78')
CURSOR_COLOR='#e87924'
PANEL_SIZE=(6.4,4.15)
compute_button=widgets.Button(description='Compute',button_style='primary',icon='play',
                              layout=widgets.Layout(width='125px'))
status=widgets.HTML('<span style="font-size:12px">Set parameters, then press Compute.</span>')
summary=widgets.HTML()
progress_view=widgets.HTML(layout=widgets.Layout(width='100%',display='none'))
progress_view.add_class('xposim-progress')
scan_image=widgets.Image(format='png',layout=widgets.Layout(width='100%',height='auto'))
plane_image=widgets.Image(format='png',layout=widgets.Layout(width='100%',height='auto'))
plane_controls=widgets.VBox(layout=widgets.Layout(width='100%',display='none'))
details_text=widgets.HTML()
details=widgets.Accordion(children=[details_text],selected_index=None)
details.set_title(0,'Model, mesh and diagnostics')


def compact_number(value):
    text=f'{float(value):.2f}'.rstrip('0').rstrip('.')
    return '0' if text=='-0' else text


TICK_FORMATTER=FuncFormatter(lambda value,position:compact_number(value))


def engineering(values, unit=''):
    """Keep ticks compact and move a common power of ten to the axis label."""
    finite=np.asarray(values,dtype=float)
    finite=finite[np.isfinite(finite)]
    maximum=float(np.max(abs(finite))) if finite.size else 0.0
    exponent=0
    if maximum>=80:
        exponent=int(np.ceil(np.log10(maximum/80)))
    elif 0<maximum<0.01:
        exponent=int(np.floor(np.log10(maximum)))
    multiplier=10.0**exponent
    suffix=(' '+r'$\times\,10^{'+str(exponent)+'}$') if exponent else ''
    return multiplier,unit+suffix


def length_units(values):
    finite=np.asarray(values,dtype=float)
    finite=finite[np.isfinite(finite)]
    maximum=float(np.max(abs(finite))) if finite.size else 0
    scale,label=(1.0,'m') if maximum>=0.1 else ((1e-3,'mm') if maximum>=1e-4 else ((1e-6,'µm') if maximum>=1e-7 else (1e-9,'nm')))
    power,suffix=engineering(finite/scale,label)
    return scale*power,suffix


def fixed_axes(ax, x=True, y=True):
    for axis,enabled in ((ax.xaxis,x),(ax.yaxis,y)):
        if enabled:
            axis.set_major_locator(MaxNLocator(nbins=4,min_n_ticks=3))
            axis.set_major_formatter(TICK_FORMATTER)
    ax.tick_params(labelsize=8)
    for spine in ax.spines.values():
        spine.set_color('#d5dce5')
    ax.set_facecolor('#ffffff')


def plot_png(figure):
    stream=io.BytesIO()
    figure.savefig(stream,format='png',dpi=140,facecolor='#ffffff')
    plt.close(figure)
    return stream.getvalue()


def metric_value(value, unit=''):
    if value is None or not np.isfinite(value):
        return 'unresolved'
    exponent=0
    if value and (abs(value)<0.1 or abs(value)>=1e4):
        exponent=int(np.floor(np.log10(abs(value))))
    text=compact_number(value/10.0**exponent)
    if exponent:
        text+=f' × 10<sup>{exponent}</sup>'
    return text+((' '+html.escape(unit)) if unit else '')


def distance_value(value):
    # Cards use physical units instead of embedding Matplotlib math markup.
    scale,unit=(1.,'m') if abs(value)>=.1 else ((1e-3,'mm') if abs(value)>=1e-4 else ((1e-6,'µm') if abs(value)>=1e-7 else (1e-9,'nm')))
    return metric_value(value/scale,unit)


def width_value(value):
    if value is None or not np.isfinite(value):
        return 'unresolved'
    scale,unit=(1e-9,'nm') if abs(value)<1e-6 else (1e-6,'µm')
    return metric_value(value/scale,unit)


def axis_values(values, formatter):
    values=tuple(values)
    if len(values)==1:
        return formatter(values[0])
    return ' · '.join('<span style="white-space:nowrap">'+axis+': '+formatter(value)+'</span>' for axis,value in zip(('x','y'),values))


def metric_card(label,value,dynamic=False):
    background='rgba(232,121,36,.055)' if dynamic else 'rgba(127,127,127,.035)'
    return '<div style="min-width:0;padding:7px 10px;border:1px solid rgba(127,127,127,.2);border-radius:8px;background:'+background+'"><div style="font-size:11px;line-height:1.25;opacity:.75;margin-bottom:4px">'+html.escape(label)+'</div><div style="font-size:15px;line-height:1.35;font-weight:600;overflow-wrap:anywhere">'+value+'</div></div>'


def update_summary(result,plane):
    parallel=result.configuration.source_type=='plane'
    image_card=(metric_card('Incident field','Plane wave') if parallel else
                metric_card('Analytical image distance · finite source',distance_value(result.analytical_image_distance_m)))
    magnification=('Not applicable · plane wave' if parallel else axis_values(plane.magnification,metric_value))
    cards=[metric_card('Numerical peak intensity distance',distance_value(result.peak_z_m)),
           metric_card('Analytical focal distance · parallel beam',distance_value(result.collimated_focus_m)),
           image_card,
           metric_card('Transmission',compact_number(result.transmission*100)+'%'),
           metric_card('Selected z · from CRL exit',distance_value(plane.z_m),True),
           metric_card('FWHM · selected z',axis_values(np.atleast_1d(plane.metrics.fwhm),width_value),True),
           metric_card('Peak intensity · selected z',metric_value(plane.metrics.max_intensity),True),
           metric_card('Magnification · selected z',magnification,True)]
    summary.value='<div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:7px;margin:5px 0 8px">'+''.join(cards)+'</div>'


def render_scan(result, selected_plane):
    zscale,zunit=length_units(result.z_m)
    xscale,xunit=length_units(result.x_m)
    peakscale,peakunit=engineering(result.peak_intensity,'a.u.')
    xwidth=result.fwhm_m if result.fwhm_m.ndim==1 else result.fwhm_m[:,0]
    width_arrays=[xwidth]
    if result.configuration.geometry=='two-dimensional':
        width_arrays.append(result.fwhm_m[:,1])
    widthscale,widthunit=length_units(np.concatenate(width_arrays))
    with plt.rc_context({'font.size':9,'axes.titlesize':11,'axes.labelsize':9}):
        figure=plt.figure(figsize=PANEL_SIZE,layout='constrained')
        spec=figure.add_gridspec(3,2,width_ratios=[1,.028],height_ratios=[2.0,1,1],hspace=.04)
        map_ax=figure.add_subplot(spec[0,0])
        peak_ax=figure.add_subplot(spec[1,0],sharex=map_ax)
        width_ax=figure.add_subplot(spec[2,0],sharex=map_ax)
        color_ax=figure.add_subplot(spec[0,1])
        top=max(float(np.max(result.longitudinal_intensity)),np.finfo(float).tiny)
        mesh=map_ax.pcolormesh(result.z_m/zscale,result.x_m/xscale,
            result.longitudinal_intensity.T/top,shading='auto',cmap=CMAP,norm=Normalize(0,1),rasterized=True)
        colorbar=figure.colorbar(mesh,cax=color_ax)
        colorbar.set_ticks([0,.5,1])
        colorbar.ax.yaxis.set_major_formatter(TICK_FORMATTER)
        colorbar.ax.tick_params(labelsize=8)
        map_ax.set(ylabel=f'x ({xunit})',title='Intensity distribution near the focus')
        peak_ax.plot(result.z_m/zscale,result.peak_intensity/peakscale,color=INTENSITY_COLOR,lw=1.7)
        peak_ax.set(ylabel='Peak intensity\n('+peakunit+')')
        for index,width in enumerate(width_arrays):
            width_ax.plot(result.z_m/zscale,width/widthscale,label='x' if index==0 else 'y',
                          color=WIDTH_COLORS[index],lw=2.3 if index==0 else 1.7,
                          ls='-' if index==0 else (0,(4,2.5)),alpha=.85 if index==0 else .95)
        if len(width_arrays)>1:
            width_ax.legend(frameon=False,fontsize=8,loc='upper right')
        width_ax.set(xlabel=f'z from CRL exit ({zunit})',ylabel=f'FWHM ({widthunit})')
        current_z=selected_plane.z_m
        for ax in (map_ax,peak_ax,width_ax):
            fixed_axes(ax)
            ax.axvline(result.peak_z_m/zscale,color='#667085',ls=':',lw=.85)
            ax.axvline(current_z/zscale,color=CURSOR_COLOR,lw=.85,alpha=.85)
        peak_ax.plot([current_z/zscale],[selected_plane.metrics.max_intensity/peakscale],'o',color=CURSOR_COLOR,ms=4)
        for width in np.atleast_1d(selected_plane.metrics.fwhm)[:len(width_arrays)]:
            if width is not None and np.isfinite(width):
                width_ax.plot([current_z/zscale],[width/widthscale],'o',color=CURSOR_COLOR,ms=4)
        state['scan_cursor_m']=current_z
        for ax in (map_ax,peak_ax):
            ax.tick_params(labelbottom=False)
        for ax in (peak_ax,width_ax):
            ax.grid(alpha=.14)
        state['scan_axes']=(map_ax,peak_ax,width_ax)
        scan_image.value=plot_png(figure)


def nearest_z(z_m):
    """Snap physical z to a computed overview plane, including the local peak."""
    positions=state['result'].z_m
    return float(positions[np.argmin(abs(positions-float(z_m)))])


def select_plane(index):
    """Finish each widget event synchronously, within its frontend comm context."""
    result=state['result']
    if result is None or state.get('updating_selection'):
        return
    index=int(np.clip(index,0,len(result.z_m)-1))
    state['updating_selection']=True
    try:
        # Mutually synchronized controls must not recursively trigger a render.
        state['slider'].value=index
        state['z_input'].value=compact_number(result.z_m[index]/state['z_unit_scale'])
        state['previous_button'].disabled=index==0
        state['next_button'].disabled=index==len(result.z_m)-1
        state['plane_label'].value=f'<span style="font-size:12px">Plane {index+1} / {len(result.z_m)}</span>'
        draw_plane(float(result.z_m[index]))
        state['displayed_index']=index
        state['error']=None
        if state.get('ready_status'):
            status.value=state['ready_status']
    except Exception as exc:
        state['error']=exc
        status.value='<span style="font-size:12px;color:#b45309">Plane update failed: '+html.escape(str(exc))+'</span>'
    finally:
        state['updating_selection']=False


def build_plane_controls(result,zscale,zunit):
    index=int(np.argmin(abs(result.z_m-result.peak_z_m)))
    slider=widgets.IntSlider(value=index,min=0,max=len(result.z_m)-1,step=1,
        description='',readout=False,continuous_update=True,
        layout=widgets.Layout(width='auto',flex='1 1 240px',min_width='180px'))
    z_input=widgets.Text(value=compact_number(result.z_m[index]/zscale),
        description=f'z ({zunit})',continuous_update=False,
        style={'description_width':'initial'},layout=widgets.Layout(width='160px'))
    previous=widgets.Button(icon='chevron-left',tooltip='Previous z plane',layout=widgets.Layout(width='34px'))
    following=widgets.Button(icon='chevron-right',tooltip='Next z plane',layout=widgets.Layout(width='34px'))
    label=widgets.HTML(layout=widgets.Layout(width='105px'))
    slider.add_class('xposim-plane-slider')
    z_input.add_class('xposim-z-input')
    previous.add_class('xposim-previous-plane')
    following.add_class('xposim-next-plane')
    label.add_class('xposim-plane-label')
    state.update(slider=slider,z_input=z_input,previous_button=previous,
                 next_button=following,plane_label=label)
    def slider_changed(change):
        select_plane(change['new'])
    def z_changed(change):
        if state.get('updating_selection'):
            return
        try:
            target=float(change['new'])*state['z_unit_scale']
            if not np.isfinite(target):
                raise ValueError('Enter a finite z value.')
            select_plane(int(np.argmin(abs(result.z_m-target))))
        except (ValueError,TypeError) as exc:
            status.value='<span style="font-size:12px;color:#b45309">'+html.escape(str(exc))+'</span>'
    slider.observe(slider_changed,names='value')
    z_input.observe(z_changed,names='value')
    previous.on_click(lambda _:select_plane(state['slider'].value-1))
    following.on_click(lambda _:select_plane(state['slider'].value+1))
    state['handler']=slider_changed
    plane_controls.children=(widgets.HBox([previous,slider,following,z_input,label],
        layout=widgets.Layout(width='100%',align_items='center',flex_flow='row wrap')),)
    select_plane(index)


def draw_plane(z_m):
    result=state['result']
    if result is None:
        return
    z_m=nearest_z(z_m)
    plane=result.plane_at(z_m)
    state['last_plane']=plane
    state['plane_updates']+=1
    peak=max(float(np.max(result.peak_intensity)),np.finfo(float).tiny)
    with plt.rc_context({'font.size':9,'axes.titlesize':10,'axes.labelsize':9}):
        figure=plt.figure(figsize=PANEL_SIZE,layout='constrained')
        if plane.y_m is not None:
            spec=figure.add_gridspec(1,2,width_ratios=[1.08,1],wspace=.08)
            map_ax=figure.add_subplot(spec[0,0])
            profile_ax=figure.add_subplot(spec[0,1])
            xscale,xunit=length_units(plane.x_m)
            yscale,yunit=length_units(plane.y_m)
            mesh=map_ax.pcolormesh(plane.x_m/xscale,plane.y_m/yscale,plane.intensity/peak,
                                  shading='auto',cmap=CMAP,norm=Normalize(0,1),rasterized=True)
            cb=figure.colorbar(mesh,ax=map_ax,location='top',fraction=.045,pad=.035)
            cb.set_ticks([0,.5,1]);cb.ax.xaxis.set_major_formatter(TICK_FORMATTER)
            cb.ax.tick_params(labelsize=8)
            map_ax.set(xlabel=f'x ({xunit})',ylabel=f'y ({yunit})',title='Transverse intensity',aspect='auto')
            j,i=np.unravel_index(int(np.argmax(plane.intensity)),plane.intensity.shape)
            profiles=[(plane.x_m,plane.intensity[j,:],'x'),
                      (plane.y_m,plane.intensity[:,i],'y')]
            map_ax.axhline(plane.y_m[j]/yscale,color=PROFILE_COLORS[0],lw=.75,alpha=.6)
            map_ax.axvline(plane.x_m[i]/xscale,color=PROFILE_COLORS[1],lw=.75,ls='--',alpha=.6)
            fixed_axes(map_ax)
            state['plane_axes']=(map_ax,profile_ax)
        else:
            profile_ax=figure.add_subplot(111)
            profiles=[(plane.x_m,plane.intensity,'x')]
            state['plane_axes']=(profile_ax,)
        scale,unit=length_units(np.concatenate([coord for coord,profile,label in profiles]))
        for index,(coord,profile,label) in enumerate(profiles):
            profile_ax.plot(coord/scale,profile/peak,label=label,
                            color=PROFILE_COLORS[index],lw=2.3 if index==0 else 1.7,
                            ls='-' if index==0 else (0,(4,2.5)),alpha=.8 if index==0 else .95,
                            zorder=2+index)
        profile_ax.set(xlabel=f'Transverse coordinate ({unit})',ylabel='I / reference peak',
                       title='Profiles through the peak' if len(profiles)>1 else 'Transverse intensity',ylim=(0,1.05))
        if len(profiles)>1:
            profile_ax.legend(frameon=False,fontsize=8,loc='upper right')
        profile_ax.grid(alpha=.15)
        fixed_axes(profile_ax)
        plane_image.value=plot_png(figure)
    if state.get('scan_cursor_m')!=plane.z_m:
        render_scan(result,plane)
    update_summary(result,plane)


STAGE_LABELS={'preparing':'Preparing calculation','grid':'Preparing CRL and grid',
    'window_checks':'Checking propagation window','source_modes':'Preparing source modes',
    'detector':'Preparing detector grid','peak_search':'Finding peak-intensity distance',
    'axial_roi':'Finding display interval','sampling':'Calculating z planes',
    'finalizing':'Finalizing beam metrics','complete':'Numerical calculation complete',
    'rendering':'Rendering plots','ready':'Complete','failed':'Calculation stopped'}


def duration_text(seconds):
    seconds=max(0,int(np.ceil(seconds)))
    minutes,seconds=divmod(seconds,60)
    hours,minutes=divmod(minutes,60)
    return (f'{hours:d}:{minutes:02d}:{seconds:02d}' if hours else f'{minutes:02d}:{seconds:02d}')


def show_progress(stage,completed=0,total=None,elapsed_s=0.0,eta_s=None,detail=''):
    label=STAGE_LABELS.get(stage,stage)
    if stage=='ready':
        progress_view.value='<div style="display:flex;align-items:center;gap:8px;font-size:11px;opacity:.8;margin:1px 0 3px"><progress max="1" value="1" aria-label="Complete" style="width:85px;height:8px;accent-color:#25864b"></progress><span>Complete · '+duration_text(elapsed_s)+'</span></div>'
        progress_view.layout.display=''
        state['progress_stage']=stage
        return
    if total is None:
        amount=f'{completed} evaluations' if stage in {'peak_search','axial_roi'} else ''
        bar='<progress aria-label="'+html.escape(label,quote=True)+'" style="width:100%;height:12px"></progress>'
    else:
        amount=f'{completed} / {total}'
        bar='<progress aria-label="'+html.escape(label,quote=True)+'" max="'+str(total)+'" value="'+str(completed)+'" style="width:100%;height:12px;accent-color:#3977d7"></progress>'
    if stage=='ready':
        remaining='Completed'
    elif stage=='failed':
        remaining='Stopped'
    elif eta_s is not None:
        remaining='ETA for this stage: ≈ '+duration_text(eta_s)
    elif total is None:
        remaining='ETA: estimating · adaptive stage' if stage in {'grid','peak_search','axial_roi'} else 'ETA: estimating'
    else:
        remaining='ETA for this stage: estimating'
    progress_view.value='<div style="padding:7px 10px;margin:3px 0 5px;border:1px solid rgba(127,127,127,.2);border-radius:8px;font-size:12px"><div style="display:flex;justify-content:space-between;gap:12px"><b>'+html.escape(label)+'</b><span>'+html.escape(amount)+'</span></div>'+bar+'<div style="display:flex;justify-content:space-between;gap:12px;flex-wrap:wrap"><span>Elapsed: '+duration_text(elapsed_s)+'</span><span>'+html.escape(remaining)+'</span></div>'+('<div style="font-size:11px;opacity:.75;margin-top:3px">'+html.escape(detail)+'</div>' if detail else '')+'</div>'
    progress_view.layout.display=''
    state['progress_stage']=stage


def on_calculation_progress(update):
    state['last_progress']=update
    show_progress(update.stage,update.completed,update.total,update.elapsed_s,update.eta_s,update.detail)


def compute_focus(_=None):
    started=time.perf_counter()
    show_progress('preparing')
    compute_button.disabled=True
    status.value='<span style="font-size:12px"><progress style="width:90px;height:8px"></progress> Computing…</span>'
    state['error']=None
    for name in ('slider','z_input','previous_button','next_button','plane_label'):
        widget=state.get(name)
        if widget is not None:
            widget.unobserve_all();widget.close()
        state[name]=None
    state['slider']=None;state['result']=None
    state.pop('last_plane',None)
    state.pop('scan_cursor_m',None)
    summary.value='';scan_image.value=b'';plane_image.value=b''
    results_panel.layout.display='none';details.layout.display='none';plane_controls.layout.display='none'
    gc.collect()
    try:
        config=read_configuration()
        result=xpos.estimate_crl_focus(config,progress=on_calculation_progress)
        elapsed=time.perf_counter()-started
        state['result']=result
        show_progress('rendering',elapsed_s=time.perf_counter()-started)
        zscale,zunit=length_units(result.z_m)
        state['z_unit_scale']=zscale
        build_plane_controls(result,zscale,zunit)
        if state['error'] is not None:
            raise state['error']
        results_panel.layout.display='flex'
        plane_controls.layout.display=''
        details.layout.display=''
        source_label=('Plane wave at CRL entrance' if config.source_type=='plane' else 'Point source' if config.source_type=='point' else 'Gaussian source · FWHM '+axis_values(
            (config.source_fwhm_x_m,) if config.grid_dimension==1 else (config.source_fwhm_x_m,config.source_fwhm_y_m),width_value))
        rows=[('Material',html.escape(config.material_formula)+' · '+metric_value(config.density_g_cm3,'g/cm³')),
              ('Surface radius / full aperture',distance_value(config.radius_m)+' / '+distance_value(config.aperture_m)),
              ('Source',source_label),
              *([] if config.source_type=='plane' else [('Source-to-CRL distance',distance_value(config.source_distance_m))]),
              ('CRL length',distance_value(result.lens_length_m)),
              ('Propagation grid',str(result.grid_shape)),
              ('Detector grid',str(result.detector_shape)),
              ('Propagation x step',distance_value(result.grid_step_m)),
              ('Detector x step',distance_value(result.detector_step_m)),
              ('Peak search evaluations',str(result.numerical_peak.evaluations))]
        detail='<div style="font-size:12px;line-height:1.6">'+''.join('<div><b>'+label+':</b> '+value+'</div>' for label,value in rows)
        detail+=''.join('<div>'+html.escape(message.partition(':')[2].strip())+'</div>' for message in result.diagnostics)
        details_text.value=detail+'</div>'
        details.set_title(0,f'Model, mesh and diagnostics ({len(result.diagnostics)})')
        device='CPU / NumPy' if result.backend=='numpy' else _cuda_device_name()
        elapsed=time.perf_counter()-started
        status.value=f'<span style="font-size:12px">{html.escape(device)} · {compact_number(elapsed)} s</span>'
        state['ready_status']=status.value
        show_progress('ready',1,1,time.perf_counter()-started,0.0)
    except Exception as exc:
        show_progress('failed',elapsed_s=time.perf_counter()-started,detail=str(exc))
        state['error']=exc;state['result']=None
        summary.value='';scan_image.value=b'';plane_image.value=b''
        plane_controls.children=()
        results_panel.layout.display='none';details.layout.display='none';plane_controls.layout.display='none'
        status.value='<span style="font-size:12px;color:#b45309">'+html.escape(str(exc))+'</span>'
    finally:
        compute_button.disabled=False


def _hardware_help():
    if xpos.select_backend('auto')=='cupy':
        return '<b>'+html.escape(_cuda_device_name())+'</b> · CuPy acceleration is active.'
    return '<b>CPU / NumPy</b> · To enable GPU in Colab: <b>Runtime → Change runtime type → GPU → Save</b>, then reconnect and run all cells. The actual CUDA device is checked.'


def _cuda_device_name():
    try:
        device=xpos.cp.cuda.Device()
        name=xpos.cp.cuda.runtime.getDeviceProperties(device.id)['name']
        if isinstance(name,bytes):name=name.decode()
        return 'CUDA GPU · '+str(name)
    except Exception:
        return 'CuPy / CUDA GPU'

compute_button.on_click(compute_focus)
hardware_notice=widgets.HTML('<div style="font-size:12px;line-height:1.25;padding:1px 0">'+_hardware_help()+'</div>')
left=widgets.VBox([scan_image],layout=widgets.Layout(flex='1 1 480px',min_width='0',width='auto'))
right=widgets.VBox([plane_image],layout=widgets.Layout(flex='1 1 480px',min_width='0',width='auto'))
results_panel=widgets.HBox([left,right],layout=widgets.Layout(width='100%',flex_flow='row wrap',align_items='stretch',gap='10px',display='none'))
details.layout.display='none'
toolbar=widgets.HBox([compute_button,status],layout=widgets.Layout(width='100%',align_items='center',gap='12px',margin='3px 0 0'))
explorer_root=widgets.VBox([hardware_notice,toolbar,progress_view,summary,results_panel,plane_controls,details],
                         layout=widgets.Layout(width='100%',max_width='1280px',margin='0 auto'))
explorer_root.add_class('xposim-explorer')
results_panel.add_class('xposim-results')
right.add_class('xposim-detector')
plane_controls.add_class('xposim-slider-below')
summary.add_class('xposim-summary')
display(explorer_root)
if not _first_explorer_initialization:
    status.value='<span style="font-size:12px">Parameters retained. Press Compute.</span>'


<details><summary>Model and numerical notes</summary>

- One-dimensional focusing works on a 1D grid or a 2D grid with finite structure depth along y. Two-dimensional focusing uses a 2D grid.
- Plane wave at CRL entrance uses uniform amplitude and phase inside the pupil. Source distance and size do not apply, and source magnification is undefined.
- Gaussian source sizes are intensity FWHM. Incoherent point sampling and Gaussian intensity convolution remain available; convolution assumes a shift-invariant pupil response.
- Fast uses the propagation-grid samples. Accurate evaluates the propagated complex Fourier field on a dense local detector grid, rather than smoothing an intensity image. Entrance phase/amplitude and propagation-window convergence still need checking.
- A local peak-intensity distance is found independently of display-z samples; it need not coincide with minimum width or the geometrical image plane. The slider snaps to calculated display planes, including the numerical maximum.
- Calculation progress reports completed batches, elapsed time and ETA for the current stage. Adaptive search/ROI lengths are unknown, so those stages show activity and evaluation counts without a fabricated percentage or ETA. No background threads or timers are used.
- Every 2D plane shows its transverse map and both x/y profiles together. Distinct colors and solid/dashed lines keep coincident curves visible. Both main panels have the same taller canvas, with one z slider directly below them. An integer plane slider, physical z input and Previous/Next buttons share a synchronous callback. No background timer or event loop is needed to update a plane.
- Intensities and axes are linear; CMRmap is white at maximum. Ticks use up to two decimals without trailing zeros, with unit/power scales.
- Parameter fields have their own output, outside the results view. They are never automatically collapsed or hidden. Rerunning their cell retains entered values in the same live kernel. Calculations are not saved.
- [Usage and model guide](https://github.com/mamichberdey/XPOSim/blob/codex/xposim-correctness-gpu-batching/COLAB_GUIDE.md)

</details>
